In [6]:
import pandas as pd
import numpy as np

# --- DATA LOADING & PROCESSING (KEEPING YOUR ORIGINAL LOGIC) ---
states = ['Colorado', 'Iowa', 'Missouri', 'Nebraska', 'Wisconsin']
base_url = "s3://hackthon15cornsdatasets/{}_Dates.csv"
weather_url = "s3://hackthon15cornsdatasets/weather_data/{}_monthly_weather.csv"
all_data = []

for state in states:
    df = pd.read_csv(base_url.format(state))
    df = df[df['Data Item'].str.contains('YIELD, MEASURED IN BU / ACRE', case=False)].copy()
    df['Period'] = df['Period'].str.strip().str.upper().replace({
        'YEAR - AUG FORECAST': 'AUG', 'YEAR - SEP FORECAST': 'SEP',
        'YEAR - OCT FORECAST': 'OCT', 'YEAR': 'FINAL'
    })
    df = df[df['Period'].isin(['AUG', 'SEP', 'OCT', 'FINAL'])].copy()
    df['Value'] = pd.to_numeric(df['Value'].astype(str).str.replace(',', ''), errors='coerce')
    yield_pivot = df.pivot_table(index='Year', columns='Period', values='Value')

    df_w = pd.read_csv(weather_url.format(state))
    rename_map = {col: col.upper() for col in df_w.columns}
    for col in df_w.columns:
        c_low = col.lower()
        if 'prcp' in c_low: rename_map[col] = 'PRCP'
        if 'tmax' in c_low: rename_map[col] = 'TMAX'
        if 'tmin' in c_low: rename_map[col] = 'TMIN'
        if 'year' in c_low: rename_map[col] = 'Year'
        if 'month' in c_low: rename_map[col] = 'Month'
    
    df_w = df_w.rename(columns=rename_map)
    df_w['TAVG'] = ((df_w['TMAX'] + df_w['TMIN']) / 2) * 9/5 + 32 

    weather_summary = df_w[df_w['Month'].isin([7, 8, 9])].pivot_table(
        index='Year', columns='Month', values=['PRCP', 'TAVG']
    )
    weather_summary.columns = [f'{col}_{month}' for col, month in weather_summary.columns]

    combined = yield_pivot.join(weather_summary, how='inner')
    combined['State'] = state
    all_data.append(combined.reset_index())

df_master = pd.concat(all_data, ignore_index=True)


def get_weather_features(state, year, time_point):
    """Extracts weather features from df_master for the analog search."""
    row = df_master[(df_master['State'] == state) & (df_master['Year'] == year)]
    if row.empty: return None
    
    if time_point == "Aug1":
        return row[['PRCP_7', 'TAVG_7']].values[0]
    elif time_point == "Sep1":
        return row[['PRCP_7', 'TAVG_7', 'PRCP_8', 'TAVG_8']].values[0]
    else: 
        return row[['PRCP_7', 'TAVG_7', 'PRCP_8', 'TAVG_8', 'PRCP_9', 'TAVG_9']].values[0]

def analog_years(state, target_year, time_point, n_analogs=5, history_start=1980):
    target_feats = get_weather_features(state, target_year, time_point)
    if target_feats is None: return None, None

    history_years = list(range(history_start, target_year))
    history_feats, valid_years = [], []
    
    for i in history_years:
        feats = get_weather_features(state, i, time_point)
        if feats is not None:
            history_feats.append(feats)
            valid_years.append(i)
            
    if not history_feats: return None, None
    history_feats = np.array(history_feats)

    mean = history_feats.mean(axis=0)
    std = history_feats.std(axis=0) + 1e-3
    z_history = (history_feats - mean) / std
    z_target = (target_feats - mean) / std

    distances = np.linalg.norm(z_history - z_target, axis=1)
    sorted_idx = np.argsort(distances)[:n_analogs]
    
    return [valid_years[i] for i in sorted_idx], distances[sorted_idx]

def compute_uncertainty_stats(state, target_year, time_point):
    """Calculates the Standard Deviation of yields from the top 5 analog years."""
    analogs, _ = analog_years(state, target_year, time_point)
    if analogs is None: return np.nan
    
    analog_yields = df_master[(df_master['State'] == state) & 
                              (df_master['Year'].isin(analogs))]['FINAL']
    
    return round(analog_yields.std(), 2) if not analog_yields.empty else np.nan


df_aug_snapshot = df_master[['State', 'Year', 'PRCP_7', 'TAVG_7']].copy()
df_aug_snapshot['Uncertainty'] = df_aug_snapshot.apply(
    lambda x: compute_uncertainty_stats(x['State'], x['Year'], "Aug1"), axis=1)

df_sep_snapshot = df_master[['State', 'Year', 'PRCP_8', 'TAVG_8']].copy()
df_sep_snapshot['Uncertainty'] = df_sep_snapshot.apply(
    lambda x: compute_uncertainty_stats(x['State'], x['Year'], "Sep1"), axis=1)

df_oct_snapshot = df_master[['State', 'Year', 'PRCP_9', 'TAVG_9']].copy()
df_oct_snapshot['Uncertainty'] = df_oct_snapshot.apply(
    lambda x: compute_uncertainty_stats(x['State'], x['Year'], "EoS"), axis=1)

# --- OUTPUT ---
print("\n--- TABLE A: YIELD TRAJECTORY ---")
print(df_master[['State', 'Year', 'AUG', 'SEP', 'OCT', 'FINAL']].head())

print("\n--- TABLE B1: AUGUST SNAPSHOT ---")
print(df_aug_snapshot.head())

print("\n--- TABLE B2: SEPTEMBER SNAPSHOT---")
print(df_sep_snapshot.head())

print("\n--- TABLE B3:  OCTOBER SNAPSHOT---")
print(df_oct_snapshot.head())


--- TABLE A: YIELD TRAJECTORY ---
      State  Year    AUG    SEP    OCT  FINAL
0  Colorado  2005  137.0  130.0  135.0  148.0
1  Colorado  2006  154.0  152.0  150.0  156.0
2  Colorado  2007  150.0  150.0  150.0  140.0
3  Colorado  2008  150.0  145.0  140.0  137.0
4  Colorado  2009  140.0  138.0  140.0  153.0

--- TABLE B1: AUGUST SNAPSHOT ---
      State  Year  PRCP_7     TAVG_7  Uncertainty
0  Colorado  2005   45.30  58.345613          NaN
1  Colorado  2006  135.42  57.566387          NaN
2  Colorado  2007  105.71  57.333548         5.66
3  Colorado  2008   56.00  56.942774         8.00
4  Colorado  2009  104.60  56.273290         8.54

--- TABLE B2: SEPTEMBER SNAPSHOT---
      State  Year  PRCP_8     TAVG_8  Uncertainty
0  Colorado  2005  105.77  53.377903          NaN
1  Colorado  2006  127.27  53.319839          NaN
2  Colorado  2007  118.26  57.029871         5.66
3  Colorado  2008  125.22  53.674323         8.00
4  Colorado  2009   52.41  54.043032         8.54

--- TABLE B3:  O